*  DSC550-T302 Data Mining (2265-1)
*  3.2 Exercise: Sentiment Analysis and Preprocessing Text
*  David Jorgensen

# 3.2 Exercise: Sentiment Analysis and Preprocessing Text

# Instructions

Download the labeled training dataset from this link: [Bag of Words Meets Bags of Popcorn](https://www.kaggle.com/c/word2vec-nlp-tutorial/data). 

**Part 1: Using the TextBlob Sentiment Analyzer**
1. Import the movie review data as a data frame and ensure that the data is loaded properly.
2. How many of each positive and negative reviews are there?
3. Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.
4. Check the accuracy of this model. Is this model better than random guessing?
5. For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).

**Part 2: Prepping Text for a Custom Model**

If you want to run your own model to classify text, it needs to be in proper form to do so. The following steps will outline a procedure to do this on the movie reviews text.
 1. Convert all text to lowercase letters.
 2. Remove punctuation and special characters from the text.
 3. Remove stop words.
 4. Apply NLTK’s PorterStemmer.
 5. Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.
 6. Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.

**Additional Comments**
 * The bag-of-words and tf-idf matrices are stored as sparse matrices because most entries are zero.
 * Each row in the bag-of-words/tf-idf matrices corresponds to a movie review.
 * The columns in the bag-of-words/tf-idf matrices correspond to unique words appearing in the movie reviews.
 * Entries in the bag-of-words matrix are the number of times a word appears in a review.
 * Entries in the tf-idf matrix are numbers representing the word importance in a review.
 * The bag-of-words/tf-idf matrices are possible feature (input) matrices for model building.
 * We will revisit this preprocessed text data to build a custom model in the future.

# Assignment

## Part 1: Using the TextBlob Sentiment Analyzer

In [9]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('labeledTrainData.tsv', sep='\t')

# Check data is loaded properly
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         25000 non-null  object
 1   sentiment  25000 non-null  int64 
 2   review     25000 non-null  object
dtypes: int64(1), object(2)
memory usage: 586.1+ KB
None


In [10]:
# Convert sentiment labels to binary values
df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'positive'})

# Count positive and negative reviews
review_counts = df['sentiment'].value_counts()
print(review_counts)

sentiment
positive    12500
negative    12500
Name: count, dtype: int64


In [11]:
# Classify movie reviews as positive or negative using TextBlob
from textblob import TextBlob
# Create a function to classify sentiment
def classify_sentiment(text):
	blob = TextBlob(text)
	if blob.sentiment.polarity >= 0:
		return 'positive'
	else:
		return 'negative'
# Apply the function to the review column
df['predicted_sentiment'] = df['review'].apply(classify_sentiment)
# Display the first few rows of the dataframe with the new column
print(df.head())

       id sentiment                                             review  \
0  5814_8  positive  With all this stuff going down at the moment w...   
1  2381_9  positive  \The Classic War of the Worlds\" by Timothy Hi...   
2  7759_3  negative  The film starts with a manager (Nicholas Bell)...   
3  3630_4  negative  It must be assumed that those who praised this...   
4  9495_8  positive  Superbly trashy and wondrously unpretentious 8...   

  predicted_sentiment  
0            positive  
1            positive  
2            negative  
3            positive  
4            negative  


In [12]:
# Compare predicted sentiment with actual sentiment
from sklearn.metrics import classification_report
print(classification_report(df['sentiment'], df['predicted_sentiment']))

              precision    recall  f1-score   support

    negative       0.89      0.42      0.57     12500
    positive       0.62      0.95      0.75     12500

    accuracy                           0.69     25000
   macro avg       0.75      0.69      0.66     25000
weighted avg       0.75      0.69      0.66     25000



Comparing the predicted sentiment with the actual sentiment using classification report from sklearn.metrics, the accuracy of the TextBlob sentiment analyzer can be evaluated. The classification report provides precision, recall, f1-score, and support for each class (positive and negative). According to the classification report, the TextBlob sentiment analyzer performs better than random guessing, as it shows higher precision and recall for both positive and negative classes compared to a random classifier which would have an accuracy of around 50%. The classification report indicates an accuracy of 69%, with precision and recall values that are higher than .5 for both classes except for the negative class where recall is 0.42, which suggests that the model is better at identifying positive reviews than negative ones. Reported F1-scores of 0.75 for positive and 0.57 for netgative reviews again indicate that the model is more effective at identifying positive reviews than negative ones.

Overall, the TextBlob sentiment analyzer demonstrates a reasonable performance in classifying movie reviews as positive or negative, with a higher accuracy in identifying positive reviews compared to negative reviews.

In [19]:
# Use another prebuilt text sentiment analyzer (VADER)
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
from joblib import Parallel, delayed

sia = SentimentIntensityAnalyzer()

# Create a function to classify sentiment using VADER
def classify_sentiment_vader(text):
	sentiment_score = sia.polarity_scores(text)['compound']
	if sentiment_score >= 0:
		return 'positive'
	else:
		return 'negative'

# Apply the function in parallel to speed up processing
df['predicted_sentiment_vader'] = Parallel(n_jobs=-1)(delayed(classify_sentiment_vader)(text) for text in df['review'])

# Display the first few rows of the dataframe with the new column
print(df.head())

# Compare predicted sentiment with actual sentiment for VADER
from sklearn.metrics import classification_report
print(classification_report(df['sentiment'], df['predicted_sentiment_vader']))

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/djorgs/nltk_data...


       id sentiment                                             review  \
0  5814_8  positive  With all this stuff going down at the moment w...   
1  2381_9  positive  \The Classic War of the Worlds\" by Timothy Hi...   
2  7759_3  negative  The film starts with a manager (Nicholas Bell)...   
3  3630_4  negative  It must be assumed that those who praised this...   
4  9495_8  positive  Superbly trashy and wondrously unpretentious 8...   

  predicted_sentiment predicted_sentiment_vader  
0            positive                  negative  
1            positive                  positive  
2            negative                  negative  
3            positive                  negative  
4            negative                  positive  
              precision    recall  f1-score   support

    negative       0.78      0.53      0.64     12500
    positive       0.65      0.85      0.74     12500

    accuracy                           0.69     25000
   macro avg       0.72      0.69    

Running the same analysis on VADER, we find the same accuracy score of 69% as TextBlob. However, all precision and recall values exceed .5 for both positive and negative classes. F1-scores of 0.74 for positive and 0.64 for negative reviews indicate that the VADER model is more effective at identifying both positive and negative reviews than the TextBlob model, which had a lower recall for negative reviews. Overall, the VADER sentiment analyzer demonstrates a better performance in classifying movie reviews as positive or negative compared to the TextBlob sentiment analyzer, with higher precision and recall values for both classes.

## Part 2: Prepping Text for a Custom Model

In [20]:
# Convert all text to lowercase
df['review'] = df['review'].str.lower()

# Remove punctuation
import string
df['review'] = df['review'].str.translate(str.maketrans('', '', string.punctuation))

# Remove stop words
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

# Perform stemming
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
df['review'] = df['review'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

# Print the first few rows of the dataframe after preprocessing
print(df.head())

[nltk_data] Downloading package stopwords to /home/djorgs/nltk_data...
Exception ignored in: <function ResourceTracker.__del__ at 0x7faa30b8a660>
Traceback (most recent call last):
  File "/home/djorgs/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/djorgs/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/djorgs/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fa6fc58e660>
Traceback (most recent call last):
  File "/home/djorgs/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/djorgs/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/djorgs/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10

       id sentiment                                             review  \
0  5814_8  positive  stuff go moment mj ive start listen music watc...   
1  2381_9  positive  classic war world timothi hine entertain film ...   
2  7759_3  negative  film start manag nichola bell give welcom inve...   
3  3630_4  negative  must assum prais film greatest film opera ever...   
4  9495_8  positive  superbl trashi wondrous unpretenti 80 exploit ...   

  predicted_sentiment predicted_sentiment_vader  
0            positive                  negative  
1            positive                  positive  
2            negative                  negative  
3            positive                  negative  
4            negative                  positive  


In [21]:
# Create a bag-of-words matrix
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['review'])

# Display the dimensions of the bag-of-words matrix
print(X_bow.shape)

# Create a term frequency-inverse document frequency (tf-idf) matrix
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['review'])

# Display the dimensions of the tf-idf matrix
print(X_tfidf.shape)

(25000, 92379)
(25000, 92379)


Both bag-of-words and tf-idf matrices were created successfully. The dimensions of both matrices are (25000, 92379), which means there are 25,000 movie reviews (rows) and 92,379 unique words (columns) in the dataset.